3.1 Automatic Story Generation

In [1]:
import requests

def generate_story(index, config):
    url = "http://localhost:11434/api/generate"
    prompt = f"Write a {config['style']} short story in the genre of {config['genre']}. Make it coherent and imaginative."

    payload = {
        "model": config["model"],
        "prompt": prompt,
        "temperature": config["temp"],
        "top_k": config["top_k"],
        "top_p": config["top_p"],
        "seed": config["seed"],
        "stream": False
    }

    response = requests.post(url, json=payload)
    story = response.json().get("response", "")
    
    filename = f"story_{index}_{config['genre'].lower()}.txt"
    with open(filename, "w") as f:
        f.write(story)
    print(f"Saved: {filename}")

# Example story configs with a fixed seed
story_configs = [
    {"genre": "Sci-Fi", "style": "futuristic", "temp": 0.8, "top_k": 50, "top_p": 0.9, "model": "qwen2.5vl:3b", "seed": 42},
    {"genre": "Horror", "style": "dark", "temp": 0.9, "top_k": 40, "top_p": 0.85, "model": "llama3.2","seed": 42},
    {"genre": "Comedy", "style": "whimsical", "temp": 1.0, "top_k": 60, "top_p": 0.95, "model": "qwen2.5vl:3b","seed": 42},
    {"genre": "Mystery", "style": "tense", "temp": 0.7, "top_k": 30, "top_p": 0.8, "model": "llama3.2","seed": 42},
    {"genre": "Fantasy", "style": "epic", "temp": 0.9, "top_k": 80, "top_p": 0.9, "model": "qwen2.5vl:3b","seed": 42},
    {"genre": "Romance", "style": "emotional", "temp": 0.6, "top_k": 25, "top_p": 0.7, "model": "llama3.2","seed": 42},
    {"genre": "Thriller", "style": "fast-paced", "temp": 0.85, "top_k": 50, "top_p": 0.8, "model": "qwen2.5vl:3b","seed": 42},
    {"genre": "Historical", "style": "formal", "temp": 0.65, "top_k": 20, "top_p": 0.75, "model": "llama3.2","seed": 42},
    {"genre": "Superhero", "style": "bold", "temp": 0.95, "top_k": 70, "top_p": 0.95, "model": "qwen2.5vl:3b","seed": 42},
    {"genre": "Satire", "style": "sarcastic", "temp": 1.1, "top_k": 100, "top_p": 0.98, "model": "llama3.2","seed": 42},
]

for i, cfg in enumerate(story_configs, 1):
    generate_story(i, cfg)


Saved: story_1_sci-fi.txt
Saved: story_2_horror.txt
Saved: story_3_comedy.txt
Saved: story_4_mystery.txt
Saved: story_5_fantasy.txt
Saved: story_6_romance.txt
Saved: story_7_thriller.txt
Saved: story_8_historical.txt
Saved: story_9_superhero.txt
Saved: story_10_satire.txt


3.1 Automatic Story Generation (Parallel)

In [ ]:
import requests
import os

def generate_story(index, config, model_name):
    url = "http://localhost:11434/api/generate"
    prompt = f"Write a {config['style']} short story in the genre of {config['genre']}. Make it coherent and imaginative."

    payload = {
        "model": model_name,
        "prompt": prompt,
        "temperature": config["temp"],
        "top_k": config["top_k"],
        "top_p": config["top_p"],
        "seed": config["seed"],
        "stream": False
    }

    response = requests.post(url, json=payload)
    story = response.json().get("response", "")

    folder = f"outputs/{model_name.lower().split(':')[0]}"
    os.makedirs(folder, exist_ok=True)

    filename = f"{folder}/story_{index}_{config['genre'].lower()}.txt"
    with open(filename, "w") as f:
        f.write(story)
    print(f"Saved: {filename}")

# Fixed seed story generation configs (you can randomize these if needed)
genres_styles = [
    ("Sci-Fi", "futuristic"),
    ("Horror", "dark"),
    ("Comedy", "whimsical"),
    ("Mystery", "tense"),
    ("Fantasy", "epic"),
    ("Romance", "emotional"),
    ("Thriller", "fast-paced"),
    ("Historical", "formal"),
    ("Superhero", "bold"),
    ("Satire", "sarcastic"),
]

def build_config(genre, style, model):
    return {
        "genre": genre,
        "style": style,
        "temp": 0.7 + 0.05 * genres_styles.index((genre, style)),  # small variation
        "top_k": 30 + 10 * genres_styles.index((genre, style)),
        "top_p": 0.75 + 0.02 * genres_styles.index((genre, style)),
        "model": model,
        "seed": 42
    }

from concurrent.futures import ThreadPoolExecutor

# Prepare task list
tasks = []
for i, (genre, style) in enumerate(genres_styles, 1):
    tasks.append((i, build_config(genre, style, "llama3.2"), "llama3.2"))
    tasks.append((i, build_config(genre, style, "qwen2.5vl:3b"), "qwen2.5vl:3b"))

# Run generation in parallel (safe with I/O-bound tasks like API calls)
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(generate_story, *task) for task in tasks]

# Optional: wait for all to finish and catch exceptions
for f in futures:
    f.result()

Saved: outputs/qwen2.5vl/story_2_horror.txt
Saved: outputs/qwen2.5vl/story_1_sci-fi.txt
Saved: outputs/llama3.2/story_2_horror.txt
Saved: outputs/llama3.2/story_1_sci-fi.txt
Saved: outputs/qwen2.5vl/story_4_mystery.txt
Saved: outputs/llama3.2/story_3_comedy.txt
Saved: outputs/llama3.2/story_4_mystery.txt
Saved: outputs/qwen2.5vl/story_5_fantasy.txt
Saved: outputs/qwen2.5vl/story_6_romance.txt
Saved: outputs/llama3.2/story_6_romance.txt
Saved: outputs/llama3.2/story_5_fantasy.txt
Saved: outputs/llama3.2/story_7_thriller.txt
Saved: outputs/llama3.2/story_8_historical.txt
Saved: outputs/llama3.2/story_9_superhero.txt
Saved: outputs/qwen2.5vl/story_3_comedy.txt
Saved: outputs/llama3.2/story_10_satire.txt
Saved: outputs/qwen2.5vl/story_8_historical.txt
Saved: outputs/qwen2.5vl/story_9_superhero.txt
Saved: outputs/qwen2.5vl/story_10_satire.txt
Saved: outputs/qwen2.5vl/story_7_thriller.txt


3.1 Automatic Story Generation (qwen 2.5)

In [1]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor


def generate_story(index, config, model_name):
    url = "http://localhost:11434/api/generate"
    prompt = f"Write a {config['style']} short story in the genre of {config['genre']}. Make it coherent and imaginative."

    payload = {
        "model": model_name,
        "prompt": prompt,
        "temperature": config["temp"],
        "top_k": config["top_k"],
        "top_p": config["top_p"],
        "seed": config["seed"],
        "stream": False
    }

    response = requests.post(url, json=payload)
    story = response.json().get("response", "")

    folder = f"outputs/{model_name.lower().split(':')[0]}"
    os.makedirs(folder, exist_ok=True)

    filename = f"{folder}/story_{index}_{config['genre'].lower()}.txt"
    with open(filename, "w") as f:
        f.write(story)
    print(f"Saved: {filename}")

# Fixed seed story generation configs (you can randomize these if needed)
genres_styles = [
    ("Sci-Fi", "futuristic"),
    ("Horror", "dark"),
    ("Comedy", "whimsical"),
    ("Mystery", "tense"),
    ("Fantasy", "epic"),
    ("Romance", "emotional"),
    ("Thriller", "fast-paced"),
    ("Historical", "formal"),
    ("Superhero", "bold"),
    ("Satire", "sarcastic"),
]

def build_config(genre, style, model):
    return {
        "genre": genre,
        "style": style,
        "temp": 0.7 + 0.05 * genres_styles.index((genre, style)),  # small variation
        "top_k": 30 + 10 * genres_styles.index((genre, style)),
        "top_p": 0.75 + 0.02 * genres_styles.index((genre, style)),
        "model": model,
        "seed": 42
    }


# Prepare task list
tasks = []
for i, (genre, style) in enumerate(genres_styles, 1):
    tasks.append((i, build_config(genre, style, "qwen2.5:3b"), "qwen2.5:3b"))

# Run generation in parallel (safe with I/O-bound tasks like API calls)
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(generate_story, *task) for task in tasks]

# Optional: wait for all to finish and catch exceptions
for f in futures:
    f.result()

Saved: outputs/qwen2.5/story_4_mystery.txt
Saved: outputs/qwen2.5/story_1_sci-fi.txt
Saved: outputs/qwen2.5/story_2_horror.txt
Saved: outputs/qwen2.5/story_3_comedy.txt
Saved: outputs/qwen2.5/story_5_fantasy.txt
Saved: outputs/qwen2.5/story_6_romance.txt
Saved: outputs/qwen2.5/story_8_historical.txt
Saved: outputs/qwen2.5/story_7_thriller.txt
Saved: outputs/qwen2.5/story_9_superhero.txt
Saved: outputs/qwen2.5/story_10_satire.txt


4.1 Automatic Story Generation EVALUATION (Based on each models' generated stories)

In [2]:
import os
import numpy as np
from llama_cpp import Llama
import math
from tqdm import tqdm

# Define paths
llama_story_dir = "outputs/llama3.2"
qwen_story_dir = "outputs/qwen2.5"

# Mapping from folder name to model path
model_map = {
    "outputs/llama3.2": "llama3.2.gguf",
    "outputs/qwen2.5": "qwen2.5:3b.gguf"
}

# Cache model objects
loaded_models = {}

def get_llm(model_path):
    if model_path not in loaded_models:
        loaded_models[model_path] = Llama(model_path=model_path, n_ctx=2048, logits_all=True)
    return loaded_models[model_path]

def compute_perplexity(text, llm, stride=512):
    enc = llm.tokenize(text.encode("utf-8"), add_bos=True)
    nlls = []
    total_tokens = 0

    for start in range(0, len(enc), stride):
        end = min(start + stride, len(enc))
        input_ids = enc[start:end]

        if len(input_ids) <= 1:
            continue

        llm.reset()
        llm.eval(input_ids[:-1])
        logits = np.array(llm.eval_logits)

        for i in range(1, len(input_ids)):
            token_id = input_ids[i]
            prev_logits = logits[i - 1]
            max_logit = np.max(prev_logits)
            log_probs = prev_logits - max_logit - np.log(np.sum(np.exp(prev_logits - max_logit)))
            nlls.append(-log_probs[token_id])

        total_tokens += len(input_ids) - 1

    if total_tokens == 0:
        return float('nan')

    avg_nll = np.sum(nlls) / total_tokens
    return math.exp(avg_nll)


# Process all stories
for story_dir in [llama_story_dir, qwen_story_dir]:
    model_path = model_map[story_dir]
    llm = get_llm(model_path)

    summary_dir = os.path.join(story_dir, "summary")
    os.makedirs(summary_dir, exist_ok=True)

    story_files = [f for f in os.listdir(story_dir) if f.endswith(".txt")]

    print(f"Processing {len(story_files)} stories from {story_dir} using model {model_path}")
    for story_file in tqdm(story_files):
        story_path = os.path.join(story_dir, story_file)
        with open(story_path, "r", encoding="utf-8") as f:
            story_text = f.read()

        ppl = compute_perplexity(story_text, llm)

        # Save result
        result_path = os.path.join(summary_dir, f"{os.path.splitext(story_file)[0]}_perplexity.txt")
        with open(result_path, "w") as f:
            f.write(f"Perplexity: {ppl:.2f}\n")


llama_model_load_from_file_impl: using device Metal (AMD Radeon Pro 5500M) - 4080 MiB free
llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from llama3.2.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:                               general.tags arr[str

Processing 10 stories from outputs/llama3.2 using model llama3.2.gguf


100%|██████████| 10/10 [08:11<00:00, 49.14s/it]
llama_model_load_from_file_impl: using device Metal (AMD Radeon Pro 5500M) - 4067 MiB free
llama_model_loader: loaded meta data with 35 key-value pairs and 434 tensors from qwen2.5:3b.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Qwen2.5 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Qwen2.5
llama_model_loader: - kv   5:                         general.size_label str              = 3B
llama_model_loader: - kv   6:     

Processing 10 stories from outputs/qwen2.5 using model qwen2.5:3b.gguf


100%|██████████| 10/10 [07:51<00:00, 47.17s/it]


In [6]:
import os
import numpy as np
from llama_cpp import Llama
import math
from tqdm import tqdm

# Define paths
llama_story_dir = "outputs/llama3.2"
qwen_story_dir = "outputs/qwen2.5"

# Mapping from folder name to model path
model_map = {
    "outputs/llama3.2": "llama3.2.gguf",
    "outputs/qwen2.5": "qwen2.5:3b.gguf"
}

# Cache model objects
loaded_models = {}

def get_llm(model_path):
    if model_path not in loaded_models:
        loaded_models[model_path] = Llama(model_path=model_path, n_ctx=2048, logits_all=True)
    return loaded_models[model_path]

def compute_perplexity(text, llm, stride=512):
    enc = llm.tokenize(text.encode("utf-8"), add_bos=True)
    nlls = []
    total_tokens = 0

    for start in range(0, len(enc), stride):
        end = min(start + stride, len(enc))
        input_ids = enc[start:end]

        if len(input_ids) <= 1:
            continue

        llm.reset()
        llm.eval(input_ids[:-1])
        logits = np.array(llm.eval_logits)

        for i in range(1, len(input_ids)):
            token_id = input_ids[i]
            prev_logits = logits[i - 1]
            max_logit = np.max(prev_logits)
            log_probs = prev_logits - max_logit - np.log(np.sum(np.exp(prev_logits - max_logit)))
            nlls.append(-log_probs[token_id])

        total_tokens += len(input_ids) - 1

    if total_tokens == 0:
        return float('nan')

    avg_nll = np.sum(nlls) / total_tokens
    return math.exp(avg_nll)


# Process all stories for each model
for story_dir in [llama_story_dir, qwen_story_dir]:
    model_path = model_map[story_dir]
    llm = get_llm(model_path)

    summary_dir = os.path.join(story_dir, "summary")
    os.makedirs(summary_dir, exist_ok=True)

    story_files = [f for f in os.listdir(story_dir) if f.endswith(".txt")]
    perplexities = []

    result_file_path = os.path.join(summary_dir, f"{os.path.basename(story_dir)}_perplexities.txt")
    with open(result_file_path, "w", encoding="utf-8") as result_file:
        print(f"\nProcessing {len(story_files)} stories from {story_dir} using model {model_path}")
        result_file.write(f"Perplexity results for model {model_path}:\n")

        for story_file in tqdm(story_files):
            story_path = os.path.join(story_dir, story_file)
            with open(story_path, "r", encoding="utf-8") as f:
                story_text = f.read()

            ppl = compute_perplexity(story_text, llm)
            perplexities.append(ppl)

            msg = f"{story_file}: Perplexity = {ppl:.2f}"
            print(msg)
            result_file.write(msg + "\n")

        avg_ppl = np.mean(perplexities)
        print(f"\nAverage Perplexity for {story_dir}: {avg_ppl:.2f}")
        result_file.write(f"\nAverage Perplexity: {avg_ppl:.2f}\n")


ggml_metal_free: deallocating
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
llama_model_load_from_file_impl: using device Metal (AMD Radeon Pro 5500M) - 4061 MiB free
llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from llama3.2.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture s


Processing 10 stories from outputs/llama3.2 using model llama3.2.gguf


 10%|█         | 1/10 [00:20<03:06, 20.77s/it]

story_8_historical.txt: Perplexity = 3.00


 20%|██        | 2/10 [00:58<04:07, 30.90s/it]

...

story_7_thriller.txt: Perplexity = 6.67


 90%|█████████ | 9/10 [04:26<00:27, 27.63s/it]

story_6_romance.txt: Perplexity = 5.17


100%|██████████| 10/10 [04:59<00:00, 29.95s/it]

story_9_superhero.txt: Perplexity = 6.84

Average Perplexity for outputs/qwen2.5: 6.04


4.1 Automatic Story Generation EVALUATION (Based On Wikitext)

In [5]:
import os
import math
import numpy as np
from llama_cpp import Llama
from datasets import load_dataset
from tqdm import tqdm

# Configuration
EVAL_DATASET = "wikitext"
EVAL_SPLIT = "test"
EVAL_TEXT_LIMIT = 100_000  # Limit number of characters to avoid long runtimes

# Model paths
model_map = {
    "llama3.2": "llama3.2.gguf",
    "qwen2.5": "qwen2.5:3b.gguf"
}

# Cache model objects
loaded_models = {}

def get_llm(model_path):
    if model_path not in loaded_models:
        loaded_models[model_path] = Llama(
            model_path=model_path,
            n_ctx=2048,
            logits_all=True
        )
    return loaded_models[model_path]

def compute_perplexity(text, llm, stride=512):
    enc = llm.tokenize(text.encode("utf-8"), add_bos=True)
    nlls = []
    total_tokens = 0

    for start in range(0, len(enc), stride):
        end = min(start + stride, len(enc))
        input_ids = enc[start:end]

        if len(input_ids) <= 1:
            continue

        llm.reset()
        llm.eval(input_ids[:-1])
        logits = np.array(llm.eval_logits)

        for i in range(1, len(input_ids)):
            token_id = input_ids[i]
            prev_logits = logits[i - 1]
            max_logit = np.max(prev_logits)
            log_probs = prev_logits - max_logit - np.log(np.sum(np.exp(prev_logits - max_logit)))
            nlls.append(-log_probs[token_id])

        total_tokens += len(input_ids) - 1

    if total_tokens == 0:
        return float('nan')

    avg_nll = np.sum(nlls) / total_tokens
    return math.exp(avg_nll)

# Load external evaluation data
print("Loading evaluation data from HuggingFace dataset...")
dataset = load_dataset(EVAL_DATASET, "wikitext-2-raw-v1", split=EVAL_SPLIT)
full_text = "\n".join(dataset["text"])
eval_text = full_text[:EVAL_TEXT_LIMIT]

# Evaluate each model
for model_name, model_path in model_map.items():
    print(f"\nEvaluating {model_name} on {EVAL_DATASET} ({EVAL_SPLIT})...")
    llm = get_llm(model_path)
    perplexity = compute_perplexity(eval_text, llm)

    # Save result
    os.makedirs(f"eval_results/{model_name}", exist_ok=True)
    result_path = f"eval_results/{model_name}/perplexity_{EVAL_DATASET}_{EVAL_SPLIT}.txt"
    with open(result_path, "w") as f:
        f.write(f"Perplexity: {perplexity:.2f}\n")

    print(f"Done: {model_name} perplexity = {perplexity:.2f}")


/Users/amirmohammad/.pyenv/versions/3.8.20/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
ggml_metal_free: deallocating
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)
ggml_metal_mem_pool_free: freeing memory pool, num heaps = 0 (total = 0)


Loading evaluation data from HuggingFace dataset...


Generating validation split: 100%|██████████| 3760/3760 [00:00<00:00, 524131.18 examples/s]
llama_model_load_from_file_impl: using device Metal (AMD Radeon Pro 5500M) - 4064 MiB free
llama_model_loader: loaded meta data with 30 key-value pairs and 255 tensors from llama3.2.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Llama 3.2 3B Instruct
llama_model_loader: - kv   3:                           general.finetune str              = Instruct
llama_model_loader: - kv   4:                           general.basename str              = Llama-3.2
llama_model_loader: - kv   5:                         general.size_label str       


Evaluating llama3.2 on wikitext (test)...


llama_model_loader: - kv  25:                      tokenizer.ggml.merges arr[str,280147]  = ["Ġ Ġ", "Ġ ĠĠĠ", "ĠĠ ĠĠ", "...
llama_model_loader: - kv  26:                tokenizer.ggml.bos_token_id u32              = 128000
llama_model_loader: - kv  27:                tokenizer.ggml.eos_token_id u32              = 128009
llama_model_loader: - kv  28:                    tokenizer.chat_template str              = {{- bos_token }}\n{%- if custom_tools ...
llama_model_loader: - kv  29:               general.quantization_version u32              = 2
llama_model_loader: - type  f32:   58 tensors
llama_model_loader: - type q4_K:  168 tensors
llama_model_loader: - type q6_K:   29 tensors
print_info: file format = GGUF V3 (latest)
print_info: file type   = Q4_K - Medium
print_info: file size   = 1.87 GiB (5.01 BPW) 
init_tokenizer: initializing tokenizer for type 2
load: control token: 128254 '<|reserved_special_token_246|>' is not marked as EOG
load: control token: 128252 '<|reserved_special_tok

Done: llama3.2 perplexity = 15.59

Evaluating qwen2.5 on wikitext (test)...


llama_model_loader: - kv  26:                      tokenizer.ggml.tokens arr[str,151936]  = ["!", "\"", "#", "$", "%", "&", "'", ...
llama_model_loader: - kv  27:                  tokenizer.ggml.token_type arr[i32,151936]  = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...
llama_model_loader: - kv  28:                      tokenizer.ggml.merges arr[str,151387]  = ["Ġ Ġ", "ĠĠ ĠĠ", "i n", "Ġ t",...
llama_model_loader: - kv  29:                tokenizer.ggml.eos_token_id u32              = 151645
llama_model_loader: - kv  30:            tokenizer.ggml.padding_token_id u32              = 151643
llama_model_loader: - kv  31:                tokenizer.ggml.bos_token_id u32              = 151643
llama_model_loader: - kv  32:               tokenizer.ggml.add_bos_token bool             = false
llama_model_loader: - kv  33:                    tokenizer.chat_template str              = {%- if tools %}\n    {{- '<|im_start|>...
llama_model_loader: - kv  34:               general.quantization_version u32   

Done: qwen2.5 perplexity = 11.41


3.2 Abstractive Text Summarization

In [2]:
import requests
import os

# Ollama API URL
OLLAMA_URL = "http://localhost:11434/api/generate"

# Models to use
models = ["llama3.2", "qwen2.5vl:3b"]

# Story directory (change if needed)
story_dir = "./"

# Loop over 10 stories
for i in range(1, 11):
    # Try to find the correct story file
    story_files = [f for f in os.listdir(story_dir) if f.startswith(f"story_{i}_") and f.endswith(".txt")]
    if not story_files:
        print(f"⚠️ Story {i} not found. Skipping...")
        continue

    story_file = story_files[0]
    with open(os.path.join(story_dir, story_file), "r") as f:
        story_text = f.read()

    # Prompt template
    prompt = (
        "Summarize the following story using simple sentences. "
        "Keep the summary short, clear, and focused on the main plot and message.\n\n"
        f"{story_text}"
    )

    # Run both models
    for model in models:
        print(f"⏳ Summarizing story {i} with model {model}...")

        response = requests.post(OLLAMA_URL, json={
            "model": model,
            "prompt": prompt,
            "temperature": 0.4,
            "top_k": 20,
            "top_p": 0.7,
            "seed": 123,
            "stream": False
        })

        # Get summary result
        result = response.json()
        summary = result.get("response", "").strip()

        # Save summary to file
        summary_file = f"summary_{i}_{model.replace('.', '').replace(':', '')}.txt"
        with open(os.path.join(story_dir, summary_file), "w") as f:
            f.write(summary)

        print(f"✅ Saved: {summary_file}")


⏳ Summarizing story 1 with model llama3.2...
✅ Saved: summary_1_llama32.txt
⏳ Summarizing story 1 with model qwen2.5vl:3b...
✅ Saved: summary_1_qwen25vl3b.txt
⏳ Summarizing story 2 with model llama3.2...
✅ Saved: summary_2_llama32.txt
⏳ Summarizing story 2 with model qwen2.5vl:3b...
✅ Saved: summary_2_qwen25vl3b.txt
⏳ Summarizing story 3 with model llama3.2...
✅ Saved: summary_3_llama32.txt
⏳ Summarizing story 3 with model qwen2.5vl:3b...
✅ Saved: summary_3_qwen25vl3b.txt
⏳ Summarizing story 4 with model llama3.2...
✅ Saved: summary_4_llama32.txt
⏳ Summarizing story 4 with model qwen2.5vl:3b...
✅ Saved: summary_4_qwen25vl3b.txt
⏳ Summarizing story 5 with model llama3.2...
✅ Saved: summary_5_llama32.txt
⏳ Summarizing story 5 with model qwen2.5vl:3b...
✅ Saved: summary_5_qwen25vl3b.txt
⏳ Summarizing story 6 with model llama3.2...
✅ Saved: summary_6_llama32.txt
⏳ Summarizing story 6 with model qwen2.5vl:3b...
✅ Saved: summary_6_qwen25vl3b.txt
⏳ Summarizing story 7 with model llama3.2...
✅

3.2 Abstractive Text Summarization (Parallel)

In [6]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

OLLAMA_URL = "http://localhost:11434/api/generate"
models = ["llama3.2", "qwen2.5vl:3b"]
base_output_dir = "outputs"  # parent folder for all models

def summarize_story(i, model):
    model_folder = os.path.join(base_output_dir, model.lower().split(':')[0])
    summary_folder = os.path.join(model_folder, "summary")
    os.makedirs(summary_folder, exist_ok=True)

    # Find story file inside model folder
    story_files = [f for f in os.listdir(model_folder) if f.startswith(f"story_{i}_") and f.endswith(".txt")]
    if not story_files:
        print(f"⚠️ Story {i} not found in {model_folder}. Skipping...")
        return
    
    story_file = story_files[0]
    story_path = os.path.join(model_folder, story_file)
    with open(story_path, "r") as f:
        story_text = f.read()

    prompt = (
        "Summarize the following story using simple sentences. "
        "Keep the summary short, clear, and focused on the main plot and message.\n\n"
        f"{story_text}"
    )

    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "temperature": 0.4,
        "top_k": 20,
        "top_p": 0.7,
        "seed": 42,
        "stream": False
    })

    if response.status_code != 200:
        print(f"❌ Request failed for story {i} model {model}: {response.status_code}")
        return

    result = response.json()
    summary = result.get("response", "").strip()

    summary_file = f"summary_{i}_{model.replace('.', '').replace(':', '')}.txt"
    summary_path = os.path.join(summary_folder, summary_file)
    with open(summary_path, "w") as f:
        f.write(summary)

    print(f"✅ Saved summary: {summary_path}")

# Prepare tasks for all story/model combos
tasks = []
for i in range(1, 11):
    for model in models:
        tasks.append((i, model))

# Run in parallel with tqdm progress bar
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(summarize_story, i, model) for i, model in tasks]

    for _ in tqdm(as_completed(futures), total=len(futures), desc="Summarizing stories"):
        pass


Summarizing stories:   5%|▌         | 1/20 [01:13<23:15, 73.47s/it]

✅ Saved summary: outputs/qwen2.5vl/summary/summary_1_qwen25vl3b.txt


Summarizing stories:  10%|█         | 2/20 [01:26<11:17, 37.65s/it]

✅ Saved summary: outputs/qwen2.5vl/summary/summary_2_qwen25vl3b.txt


Summarizing stories:  15%|█▌        | 3/20 [02:09<11:28, 40.49s/it]

...

✅ Saved summary: outputs/qwen2.5vl/summary/summary_8_qwen25vl3b.txt


Summarizing stories:  95%|█████████▌| 19/20 [12:14<00:23, 23.64s/it]

✅ Saved summary: outputs/qwen2.5vl/summary/summary_9_qwen25vl3b.txt


Summarizing stories: 100%|██████████| 20/20 [12:25<00:00, 37.29s/it]

✅ Saved summary: outputs/qwen2.5vl/summary/summary_10_qwen25vl3b.txt


Added Qwen2.5:3b for summary generation

In [8]:
import requests
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

OLLAMA_URL = "http://localhost:11434/api/generate"
models = ["llama3.2", "qwen2.5vl:3b", "qwen2.5:3b"]
base_output_dir = "outputs"  # parent folder for all models

def summarize_story(i, model):
    model_folder = os.path.join(base_output_dir, model.lower().split(':')[0])
    summary_folder = os.path.join(model_folder, "summary")
    os.makedirs(summary_folder, exist_ok=True)

    # Find story file inside model folder
    story_files = [f for f in os.listdir(model_folder) if f.startswith(f"story_{i}_") and f.endswith(".txt")]
    if not story_files:
        print(f"⚠️ Story {i} not found in {model_folder}. Skipping...")
        return

    story_file = story_files[0]
    story_path = os.path.join(model_folder, story_file)
    
    # Determine summary filename
    summary_file = f"summary_{i}_{model.replace('.', '').replace(':', '')}.txt"
    summary_path = os.path.join(summary_folder, summary_file)
    
    # Skip if already exists
    if os.path.exists(summary_path):
        print(f"⏩ Already exists, skipping: {summary_path}")
        return

    with open(story_path, "r", encoding="utf-8") as f:
        story_text = f.read()

    prompt = (
        "Summarize the following story using simple sentences. "
        "Keep the summary short, clear, and focused on the main plot and message.\n\n"
        f"{story_text}"
    )

    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "temperature": 0.4,
        "top_k": 20,
        "top_p": 0.7,
        "seed": 42,
        "stream": False
    })

    if response.status_code != 200:
        print(f"❌ Request failed for story {i} model {model}: {response.status_code}")
        return

    result = response.json()
    summary = result.get("response", "").strip()

    with open(summary_path, "w", encoding="utf-8") as f:
        f.write(summary)

    print(f"✅ Saved summary: {summary_path}")

# Prepare tasks for all story/model combos
tasks = [(i, model) for i in range(1, 11) for model in models]

# Run in parallel with tqdm progress bar
with ThreadPoolExecutor(max_workers=4) as executor:
    futures = [executor.submit(summarize_story, i, model) for i, model in tasks]

    for _ in tqdm(as_completed(futures), total=len(futures), desc="Summarizing stories"):
        pass


⏩ Already exists, skipping: outputs/llama3.2/summary/summary_1_llama32.txt
⏩ Already exists, skipping: outputs/llama3.2/summary/summary_2_llama32.txt


Summarizing stories:   0%|          | 0/30 [00:00<?, ?it/s]

⏩ Already exists, skipping: outputs/qwen2.5vl/summary/summary_1_qwen25vl3b.txt
⏩ Already exists, skipping: outputs/qwen2.5vl/summary/summary_2_qwen25vl3b.txt
⏩ Already exists, skipping: outputs/llama3.2/summary/summary_3_llama32.txt
⏩ Already exists, skipping: outputs/qwen2.5vl/summary/summary_3_qwen25vl3b.txt
⏩ Already exists, skipping: outputs/llama3.2/summary/summary_4_llama32.txt
⏩ Already exists, skipping: outputs/qwen2.5vl/summary/summary_4_qwen25vl3b.txt


Summarizing stories:  30%|███       | 9/30 [00:52<02:02,  5.83s/it]

✅ Saved summary: outputs/qwen2.5/summary/summary_3_qwen253b.txt
⏩ Already exists, skipping: outputs/llama3.2/summary/summary_5_llama32.txt
⏩ Already exists, skipping: outputs/qwen2.5vl/summary/summary_5_qwen25vl3b.txt


...

✅ Saved summary: outputs/qwen2.5/summary/summary_8_qwen253b.txt


Summarizing stories:  97%|█████████▋| 29/30 [05:00<00:15, 15.20s/it]

✅ Saved summary: outputs/qwen2.5/summary/summary_9_qwen253b.txt


Summarizing stories: 100%|██████████| 30/30 [05:05<00:00, 10.19s/it]

✅ Saved summary: outputs/qwen2.5/summary/summary_10_qwen253b.txt


4.2 Abstractive Text Summarization EVALUATION

In [ ]:
import os
import csv
from rouge_score import rouge_scorer
from rich.table import Table
from rich.console import Console

story_dir = "./outputs/"  # directory where stories and summaries are saved
models = ["llama32", "qwen25vl3b", "qwen253b"]  # model names should match exactly as filenames

scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)

results = []

for i in range(1, 11):
    # Find the story file (assuming pattern story_{i}_*.txt)
    # Find the story file for each model folder and use the first found
    story_files = []
    for model_folder in ["llama3.2", "qwen2.5vl", "qwen2.5"]:
        folder_path = os.path.join(story_dir, model_folder)
        if os.path.isdir(folder_path):
            files = [f for f in os.listdir(folder_path) if f.startswith(f"story_{i}_") and f.endswith(".txt")]
            if files:
                story_files = files
                story_dir_model = folder_path
                break
    if not story_files:
        print(f"⚠️ Story {i} not found in any model folder. Skipping...")
        continue

    story_file = story_files[0]
    with open(os.path.join(story_dir_model, story_file), "r") as f:
        story_text = f.read()
    if not story_files:
        print(f"⚠️ Story {i} not found. Skipping...")
        continue

    story_file = story_files[0]
    with open(os.path.join(story_dir_model, story_file), "r") as f:
        story_text = f.read()

    for model in models:
        # Map model name to folder and summary filename pattern
        if model == "llama32":
            model_folder = "llama3.2"
        elif model == "qwen25vl3b":
            model_folder = "qwen2.5vl"
        elif model == "qwen253b":
            model_folder = "qwen2.5"
        else:
            print(f"⚠️ Unknown model: {model}. Skipping...")
            continue

        summary_folder = os.path.join(story_dir, model_folder, "summary")
        summary_file = f"summary_{i}_{model}.txt"
        summary_path = os.path.join(summary_folder, summary_file)

        if not os.path.exists(summary_path):
            print(f"⚠️ Summary file {summary_file} not found in {summary_folder}. Skipping...")
            continue

        with open(summary_path, "r") as f:
            summary_text = f.read()

        scores = scorer.score(story_text, summary_text)
        rouge1_f1 = scores['rouge1'].fmeasure

        print(f"Story {i}, Model {model}: ROUGE-1 F1 = {rouge1_f1:.4f}")
        results.append((i, model, rouge1_f1))

        if not os.path.exists(summary_path):
            print(f"⚠️ Summary file {summary_file} not found. Skipping...")
            continue

        with open(summary_path, "r") as f:
            summary_text = f.read()

        scores = scorer.score(story_text, summary_text)
        rouge1_f1 = scores['rouge1'].fmeasure

        # Create a table
        table = Table(title="ROUGE-1 F1 Scores for Story Summaries")
        table.add_column("Story", justify="right")
        table.add_column("Model", justify="left")
        table.add_column("ROUGE-1 F1", justify="right")

        for story_num, model_name, rouge1_f1 in results:
            table.add_row(str(story_num), model_name, f"{rouge1_f1:.4f}")

        console = Console()
        console.print(table)

# Optionally, save results to CSV

with open("rouge1_scores.csv", "w", newline="") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["Story", "Model", "ROUGE-1_F1"])
    writer.writerows(results)


Story 1, Model llama32: ROUGE-1 F1 = 0.2470


  ROUGE-1 F1 Scores for Story   
           Summaries            
┏━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Story ┃ Model   ┃ ROUGE-1 F1 ┃
┡━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━━━┩
│     1 │ llama32 │     0.2470 │
└───────┴─────────┴────────────┘

Story 1, Model qwen25vl3b: ROUGE-1 F1 = 0.1044


    ROUGE-1 F1 Scores for Story    
             Summaries             
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Story ┃ Model      ┃ ROUGE-1 F1 ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│     1 │ llama32    │     0.2470 │
│     1 │ qwen25vl3b │     0.1044 │
└───────┴────────────┴────────────┘

Story 1, Model qwen253b: ROUGE-1 F1 = 0.2310


...

    ROUGE-1 F1 Scores for Story    
             Summaries             
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Story ┃ Model      ┃ ROUGE-1 F1 ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│     1 │ llama32    │     0.2470 │
│     1 │ qwen25vl3b │     0.1044 │
│     1 │ qwen253b   │     0.2310 │
│     2 │ llama32    │     0.3654 │
│     2 │ qwen25vl3b │     0.2387 │
│     2 │ qwen253b   │     0.1194 │
│     3 │ llama32    │     0.3759 │
│     3 │ qwen25vl3b │     0.0940 │
│     3 │ qwen253b   │     0.0901 │
│     4 │ llama32    │     0.2466 │
│     4 │ qwen25vl3b │     0.0904 │
│     4 │ qwen253b   │     0.1191 │
│     5 │ llama32    │     0.3273 │
│     5 │ qwen25vl3b │     0.1545 │
│     5 │ qwen253b   │     0.1166 │
│     6 │ llama32    │     0.2243 │
│     6 │ qwen25vl3b │     0.0818 │
│     6 │ qwen253b   │     0.1476 │
│     7 │ llama32    │     0.2862 │
│     7 │ qwen25vl3b │     0.0674 │
│     7 │ qwen253b   │     0.1133 │
│     8 │ llama32    │     0.2910 │
│     8 │ qwen25vl3b │     0.1507 │
│     8 │ qwen253b   │     0.0940 │
│     9 │ llama32    │     0.2846 │
│     9 │ qwen25vl3b │     0.1534 │
│     9 │ qwen253b   │     0.1053 │
│    10 │ llama32    │     0.3333 │
└───────┴────────────┴────────────┘

Story 10, Model qwen25vl3b: ROUGE-1 F1 = 0.2133


    ROUGE-1 F1 Scores for Story    
             Summaries             
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Story ┃ Model      ┃ ROUGE-1 F1 ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│     1 │ llama32    │     0.2470 │
│     1 │ qwen25vl3b │     0.1044 │
│     1 │ qwen253b   │     0.2310 │
│     2 │ llama32    │     0.3654 │
│     2 │ qwen25vl3b │     0.2387 │
│     2 │ qwen253b   │     0.1194 │
│     3 │ llama32    │     0.3759 │
│     3 │ qwen25vl3b │     0.0940 │
│     3 │ qwen253b   │     0.0901 │
│     4 │ llama32    │     0.2466 │
│     4 │ qwen25vl3b │     0.0904 │
│     4 │ qwen253b   │     0.1191 │
│     5 │ llama32    │     0.3273 │
│     5 │ qwen25vl3b │     0.1545 │
│     5 │ qwen253b   │     0.1166 │
│     6 │ llama32    │     0.2243 │
│     6 │ qwen25vl3b │     0.0818 │
│     6 │ qwen253b   │     0.1476 │
│     7 │ llama32    │     0.2862 │
│     7 │ qwen25vl3b │     0.0674 │
│     7 │ qwen253b   │     0.1133 │
│     8 │ llama32    │     0.2910 │
│     8 │ qwen25vl3b │     0.1507 │
│     8 │ qwen253b   │     0.0940 │
│     9 │ llama32    │     0.2846 │
│     9 │ qwen25vl3b │     0.1534 │
│     9 │ qwen253b   │     0.1053 │
│    10 │ llama32    │     0.3333 │
│    10 │ qwen25vl3b │     0.2133 │
└───────┴────────────┴────────────┘

Story 10, Model qwen253b: ROUGE-1 F1 = 0.1077


    ROUGE-1 F1 Scores for Story    
             Summaries             
┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Story ┃ Model      ┃ ROUGE-1 F1 ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━┩
│     1 │ llama32    │     0.2470 │
│     1 │ qwen25vl3b │     0.1044 │
│     1 │ qwen253b   │     0.2310 │
│     2 │ llama32    │     0.3654 │
│     2 │ qwen25vl3b │     0.2387 │
│     2 │ qwen253b   │     0.1194 │
│     3 │ llama32    │     0.3759 │
│     3 │ qwen25vl3b │     0.0940 │
│     3 │ qwen253b   │     0.0901 │
│     4 │ llama32    │     0.2466 │
│     4 │ qwen25vl3b │     0.0904 │
│     4 │ qwen253b   │     0.1191 │
│     5 │ llama32    │     0.3273 │
│     5 │ qwen25vl3b │     0.1545 │
│     5 │ qwen253b   │     0.1166 │
│     6 │ llama32    │     0.2243 │
│     6 │ qwen25vl3b │     0.0818 │
│     6 │ qwen253b   │     0.1476 │
│     7 │ llama32    │     0.2862 │
│     7 │ qwen25vl3b │     0.0674 │
│     7 │ qwen253b   │     0.1133 │
│     8 │ llama32    │     0.2910 │
│     8 │ qwen25vl3b │     0.1507 │
│     8 │ qwen253b   │     0.0940 │
│     9 │ llama32    │     0.2846 │
│     9 │ qwen25vl3b │     0.1534 │
│     9 │ qwen253b   │     0.1053 │
│    10 │ llama32    │     0.3333 │
│    10 │ qwen25vl3b │     0.2133 │
│    10 │ qwen253b   │     0.1077 │
└───────┴────────────┴────────────┘

3.3 Natural Language Inference

In [13]:
import csv
import requests
import os
import time
from collections import defaultdict


# Required
OLLAMA_URL = "http://localhost:11434/api/generate"
models = ["llama3.2", "qwen2.5vl:3b", "qwen2.5:3b"]
DATA_FILE = "./nlp-hw4-datasets/nli/nli.csv"
OUTPUT_FILE = "./predictions/nli_predictions_compare.csv"

# Few-shot examples
few_shot_examples = """
Examples:
Premise: A woman is cooking in a kitchen.
Hypothesis: A woman is preparing food.
Answer: entailment

Premise: A dog is sleeping on the bed.
Hypothesis: The dog is outside playing fetch.
Answer: contradiction

Premise: A man is driving a car.
Hypothesis: The man is listening to music.
Answer: neutral
"""

# Prompt builder
def build_prompt(premise, hypothesis, few_shot=False):
    base_instruction = """
Determine the relationship between the following premise and hypothesis.
If the hypothesis must be true, answer "entailment".
If the hypothesis must be false, answer "contradiction".
If it neither follows nor conflicts, answer "neutral".
"""
    task = f"\nPremise: {premise}\nHypothesis: {hypothesis}\nAnswer:"
    if few_shot:
        return base_instruction + few_shot_examples + "\nNow, classify this pair:" + task
    else:
        return base_instruction + task

# Read data
with open(DATA_FILE, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    rows = list(reader)

# Compare predictions
results = []
timings = defaultdict(list)

for row in rows:
    for model in models:
        print(f"⏳ Predicting: {row['hypothesis']} with {model}...")
        
        for mode in ["zero", "few"]:  # test both
            prompt = build_prompt(row["premise"], row["hypothesis"], few_shot=(mode == "few"))
            start_time = time.time()

            response = requests.post(OLLAMA_URL, json={
                "model": model,
                "prompt": prompt,
                "temperature": 0.0,
                "stream": False
            })
            end_time = time.time()
            duration = end_time - start_time
            timings[(model, mode)].append(duration)

            result = response.json().get("response", "").strip().lower()
            result = result.split()[0]

            # Normalize output
            if "entail" in result:
                label = "entailment"
            elif "contradict" in result:
                label = "contradiction"
            elif "neutral" in result:
                label = "neutral"
            else:
                label = "unknown"

            results.append({
                "premise": row["premise"],
                "hypothesis": row["hypothesis"],
                "model": model,
                "mode": mode,
                "prediction": label
            })

            print(f"[{mode.upper()}] {label.upper()} → Took {duration:.2f} sec")

# Save to CSV
os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
with open(OUTPUT_FILE, "w", newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=["premise", "hypothesis", "model", "mode", "prediction"])
    writer.writeheader()
    writer.writerows(results)

print(f"\n✅ Saved zero-shot vs. few-shot predictions to {OUTPUT_FILE}")
# Report average inference times
print("📊 Average Inference Times:")
for (model, mode), durations in timings.items():
    avg_time = sum(durations) / len(durations)
    print(f"{model:<15} | {mode:<5} | {avg_time:.2f} seconds avg over {len(durations)} samples")



⏳ Predicting: Nobody is standing with llama3.2...
[ZERO] NEUTRAL → Took 7.07 sec
[FEW] UNKNOWN → Took 7.80 sec
⏳ Predicting: Nobody is standing with qwen2.5vl:3b...
[ZERO] CONTRADICTION → Took 2.18 sec
[FEW] CONTRADICTION → Took 2.44 sec
⏳ Predicting: Nobody is standing with qwen2.5:3b...
[ZERO] CONTRADICTION → Took 9.96 sec
[FEW] CONTRADICTION → Took 2.47 sec
⏳ Predicting: tall humans standing with llama3.2...
[ZERO] NEUTRAL → Took 5.86 sec
[FEW] UNKNOWN → Took 9.33 sec
⏳ Predicting: tall humans standing with qwen2.5vl:3b...
[ZERO] NEUTRAL → Took 0.85 sec
[FEW] NEUTRAL → Took 2.59 sec
⏳ Predicting: tall humans standing with qwen2.5:3b...
[ZERO] UNKNOWN → Took 33.09 sec
[FEW] NEUTRAL → Took 12.34 sec
⏳ Predicting: Humans standing with llama3.2...
[ZERO] NEUTRAL → Took 7.26 sec
[FEW] UNKNOWN → Took 15.08 sec
⏳ Predicting: Humans standing with qwen2.5vl:3b...
[ZERO] ENTAILMENT → Took 2.24 sec
[FEW] ENTAILMENT → Took 2.55 sec
⏳ Predicting: Humans standing with qwen2.5:3b...
[ZERO] UNKNOWN

3.3 Natural Language Inference (Parallel)

In [25]:
import csv
import requests
import os
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from rich.console import Console
from rich.table import Table

OLLAMA_URL = "http://localhost:11434/api/generate"
models = ["llama3.2", "qwen2.5vl:3b", "qwen2.5:3b"]
DATA_FILE = "./nlp-hw4-datasets/nli/nli.csv"
OUTPUT_FILE = "./predictions/nli_predictions_compare_v2.csv"

few_shot_examples = """
Examples:
Premise: A woman is cooking in a kitchen.
Hypothesis: A woman is preparing food.
Answer: entailment

Premise: A dog is sleeping on the bed.
Hypothesis: The dog is outside playing fetch.
Answer: contradiction

Premise: A man is driving a car.
Hypothesis: The man is listening to music.
Answer: neutral
"""

def build_prompt(premise, hypothesis, few_shot=False):
    base_instruction = """
Determine the relationship between the following premise and hypothesis.
If the hypothesis must be true, answer "entailment".
If the hypothesis must be false, answer "contradiction".
If it neither follows nor conflicts, answer "neutral".
"""
    task = f"\nPremise: {premise}\nHypothesis: {hypothesis}\nAnswer:"
    if few_shot:
        return base_instruction + few_shot_examples + "\nNow, classify this pair:" + task
    else:
        return base_instruction + task

def predict_for_row(row, model, mode):
    premise = row["premise"]
    hypothesis = row["hypothesis"]
    prompt = build_prompt(premise, hypothesis, few_shot=(mode == "few"))

    start_time = time.time()
    response = requests.post(OLLAMA_URL, json={
        "model": model,
        "prompt": prompt,
        "temperature": 0.0,
        "stream": False
    })
    duration = time.time() - start_time

    result = response.json().get("response", "").strip().lower()

    if "entail" in result:
        label = "entailment"
    elif "contradict" in result:
        label = "contradiction"
    elif "neutral" in result:
        label = "neutral"
    else:
        print(f"⚠️ Unrecognized response: {result}")
        label = "unknown"

    print(f"[{mode.upper()}] {model} Prediction: {label.upper()} → Took {duration:.2f} sec")
    return {
        "premise": premise,
        "hypothesis": hypothesis,
        "model": model,
        "mode": mode,
        "prediction": label,
        "duration": duration
    }

def main():
    with open(DATA_FILE, newline='') as csvfile:
        reader = csv.DictReader(csvfile)
        rows = list(reader)

    timings = defaultdict(list)
    results = []

    # Use ThreadPoolExecutor for parallel requests
    with ThreadPoolExecutor(max_workers=10) as executor:
        futures = []
        for row in rows:
            for model in models:
                for mode in ["zero", "few"]:
                    futures.append(executor.submit(predict_for_row, row, model, mode))

        for future in as_completed(futures):
            res = future.result()
            results.append(res)
            timings[(res["model"], res["mode"])].append(res["duration"])

    # Save results with duration
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w", newline='') as csvfile:
        fieldnames = ["premise", "hypothesis", "model", "mode", "prediction", "duration"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)

        # Append average times at the end as summary rows
        writer.writerow({})
        writer.writerow({"premise": "AVERAGE TIMINGS"})
        for (model, mode), durations in timings.items():
            avg_time = sum(durations) / len(durations)
            writer.writerow({
                "premise": "",
                "hypothesis": "",
                "model": model,
                "mode": mode,
                "prediction": f"{avg_time:.2f} sec avg over {len(durations)} samples",
                "duration": ""
            })

    # Print summary table with rich
    console = Console()
    table = Table(title="Zero-shot vs Few-shot Prediction Times")

    table.add_column("Model", justify="left")
    table.add_column("Mode", justify="left")
    table.add_column("Average Time (sec)", justify="right")
    table.add_column("Num Samples", justify="right")

    for (model, mode), durations in sorted(timings.items()):
        avg_time = sum(durations) / len(durations)
        table.add_row(model, mode, f"{avg_time:.2f}", str(len(durations)))

    console.print(table)
    print(f"\n✅ Saved predictions and timing info to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()


[FEW] qwen2.5:3b Prediction: CONTRADICTION → Took 17.52 sec
[ZERO] llama3.2 Prediction: NEUTRAL → Took 29.31 sec
[ZERO] qwen2.5:3b Prediction: CONTRADICTION → Took 36.15 sec
[ZERO] llama3.2 Prediction: NEUTRAL → Took 40.98 sec
[FEW] llama3.2 Prediction: ENTAILMENT → Took 70.41 sec
[ZERO] qwen2.5vl:3b Prediction: CONTRADICTION → Took 74.19 sec[FEW] qwen2.5vl:3b Prediction: NEUTRAL → Took 74.17 sec

[FEW] llama3.2 Prediction: CONTRADICTION → Took 81.02 sec
[FEW] qwen2.5vl:3b Prediction: CONTRADICTION → Took 83.12 sec[ZERO] qwen2.5vl:3b Prediction: NEUTRAL → Took 83.10 sec

[ZERO] qwen2.5vl:3b Prediction: ENTAILMENT → Took 24.31 sec
[FEW] qwen2.5vl:3b Prediction: ENTAILMENT → Took 22.39 sec
[ZERO] qwen2.5vl:3b Prediction: NEUTRAL → Took 11.93 sec
[FEW] qwen2.5vl:3b Prediction: NEUTRAL → Took 24.68 sec
[ZERO] llama3.2 Prediction: NEUTRAL → Took 89.60 sec
[FEW] qwen2.5:3b Prediction: ENTAILMENT → Took 98.96 sec
[ZERO] qwen2.5:3b Prediction: ENTAILMENT → Took 71.89 sec
[ZERO] qwen2.5vl:3b Pr

          Zero-shot vs Few-shot Prediction Times          
┏━━━━━━━━━━━━━━┳━━━━━━┳━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Model        ┃ Mode ┃ Average Time (sec) ┃ Num Samples ┃
┡━━━━━━━━━━━━━━╇━━━━━━╇━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ llama3.2     │ few  │             159.51 │         100 │
│ llama3.2     │ zero │             103.14 │         100 │
│ qwen2.5:3b   │ few  │              91.72 │         100 │
│ qwen2.5:3b   │ zero │             114.27 │         100 │
│ qwen2.5vl:3b │ few  │              13.87 │         100 │
│ qwen2.5vl:3b │ zero │              13.96 │         100 │
└──────────────┴──────┴────────────────────┴─────────────┘


✅ Saved predictions and timing info to ./predictions/nli_predictions_compare_v2.csv


4.3 Natural Language Inference EVALUATION

In [28]:
import csv
from collections import defaultdict
from rich.console import Console
from rich.table import Table

# --- Configuration ---
PREDICTIONS_FILE = "./predictions/nli_predictions_compare_v2.csv"
GOLD_FILE = "./nlp-hw4-datasets/nli/nli.csv"

# --- Data Loading ---

# Read gold labels into a dictionary for quick lookup
try:
    with open(GOLD_FILE, newline='', encoding='utf-8') as f:
        gold_rows = list(csv.DictReader(f))
        gold_dict = {(row["premise"], row["hypothesis"]): row["label"].strip().lower() for row in gold_rows}
except FileNotFoundError:
    print(f"Error: Gold standard file not found at '{GOLD_FILE}'")
    exit()

# Read model predictions
try:
    with open(PREDICTIONS_FILE, newline='', encoding='utf-8') as f:
        pred_rows = list(csv.DictReader(f))
except FileNotFoundError:
    print(f"Error: Predictions file not found at '{PREDICTIONS_FILE}'")
    exit()

# --- Evaluation ---

# Initialize a nested dictionary to store statistics for each model and mode
stats = defaultdict(lambda: {"correct": 0, "total": 0})

# Iterate through each prediction and compare it against the gold standard
for row in pred_rows:
    # Create a unique key for each premise-hypothesis pair
    key = (row["premise"], row["hypothesis"])
    
    # Normalize prediction and metadata for consistency
    pred = row["prediction"].strip().lower()
    model = row["model"].strip()
    mode = row["mode"].strip().lower()
    
    # Retrieve the gold label using the key
    gold = gold_dict.get(key)

    # If a corresponding gold label exists, update the statistics
    if gold is not None:
        stats[(model, mode)]["total"] += 1
        if pred == gold:
            stats[(model, mode)]["correct"] += 1

# --- Display Results ---

# Initialize Rich console and table for formatted output
console = Console()
table = Table(title="📊 NLI Accuracy by Model and Inference Mode", show_header=True, header_style="bold magenta")

# Define table columns
table.add_column("Model", style="cyan", no_wrap=True)
table.add_column("Mode", style="green")
table.add_column("Accuracy (%)", justify="right", style="bold yellow")
table.add_column("Correct/Total", justify="center")

# Populate the table with results
for (model, mode), result in sorted(stats.items()):
    # Calculate accuracy, handling division by zero
    accuracy = (result["correct"] / result["total"] * 100) if result["total"] > 0 else 0
    
    # Format the 'Correct/Total' column
    correct_total_str = f"{result['correct']}/{result['total']}"
    
    # Add a row to the table
    table.add_row(
        model,
        f"{mode}-shot",
        f"{accuracy:.2f}",
        correct_total_str
    )

# Print the final table to the console
console.print(table)


        📊 NLI Accuracy by Model and Inference Mode        
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Model        ┃ Mode      ┃ Accuracy (%) ┃ Correct/Total ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ llama3.2     │ few-shot  │        16.00 │    16/100     │
│ llama3.2     │ zero-shot │        21.00 │    21/100     │
│ qwen2.5:3b   │ few-shot  │        18.00 │    18/100     │
│ qwen2.5:3b   │ zero-shot │        10.00 │    10/100     │
│ qwen2.5vl:3b │ few-shot  │        32.00 │    32/100     │
│ qwen2.5vl:3b │ zero-shot │        32.00 │    32/100     │
└──────────────┴───────────┴──────────────┴───────────────┘

3.4 Image Captioning

In [6]:
import requests
import base64
import os
from PIL import Image
from io import BytesIO

# Ollama vision model
MODEL = "qwen2.5vl:3b"
OLLAMA_URL = "http://localhost:11434/api/generate"

# Directory of your image dataset
image_dir = "./nlp-hw4-datasets/ic/images"

# Output file to store generated captions
output_file = "generated_captions.txt"

# Collect image files
image_files = [f for f in os.listdir(image_dir) if f.lower().endswith(".jpg")]

# Prompt prefix
prompt = "Describe this image in one natural sentence. Be accurate and specific."

# Open output log
with open(output_file, "w") as out_f:
    for img_name in image_files:
        image_path = os.path.join(image_dir, img_name)

        # Open and convert image to base64
        with open(image_path, "rb") as img_file:
            image_bytes = img_file.read()
            image_base64 = base64.b64encode(image_bytes).decode("utf-8")

        # Prepare Ollama payload
        payload = {
            "model": MODEL,
            "prompt": prompt,
            "images": [image_base64],
            "temperature": 0.5,
            "stream": False,
            "seed":42
        }

        # Call Ollama
        print(f"⏳ Generating caption for {img_name}...")
        response = requests.post(OLLAMA_URL, json=payload)
        caption = response.json().get("response", "").strip()

        # Write result
        out_f.write(f"{img_name}\t{caption}\n")
        print(f"✅ Caption: {caption}")


⏳ Generating caption for ic-039.jpg...
✅ Caption: The image shows a red wall with a white toilet and a red sign above it that reads "DON KABHI TUM SABOOT TUMI CHODTDA - PLEASE FLUSH" in English, along with the date "23rd December" and the logo for the movie "DON 2".
⏳ Generating caption for ic-005.jpg...
✅ Caption: A plate of waffles topped with sliced bananas and a small bowl of whipped cream, accompanied by a side of bacon and a cup of syrup.
⏳ Generating caption for ic-011.jpg...
✅ Caption: The image shows a classroom setup with multiple laptops on wooden desks, each accompanied by a mouse and a small bowl of fruit, and a large projection screen on the wall.
⏳ Generating caption for ic-010.jpg...
✅ Caption: A slice of pizza with broccoli and cheese is placed on a white plate, with a wooden surface and a pepper shaker in the background.
⏳ Generating caption for ic-004.jpg...
✅ Caption: In a black-and-white photograph, two men are seated on a park bench, engaged in a conversation, wit

4.4 Image Captioning EVALUATION

In [27]:
import csv
import ast
from pycocoevalcap.cider.cider import Cider
from rich.console import Console
from rich.table import Table

# Load generated captions from generated_captions.txt (tab-separated)
gen_captions = {}
with open("generated_captions.txt", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        image_id, caption = line.split("\t", 1)  # split on TAB
        gen_captions[image_id.strip()] = caption.strip()

# Load reference captions from CSV where captions are stored as string representations of lists
gts = {}
with open("./nlp-hw4-datasets/ic/ic.csv", "r", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        image_id = row["image"].strip()
        captions_str = row["human_captions"].strip()
        captions_list = ast.literal_eval(captions_str)
        gts[image_id] = [caption.strip() for caption in captions_list if caption.strip()]

# Prepare results dict with generated captions (only for images with references)
res = {}
for img_id in gen_captions:
    if img_id in gts:
        res[img_id] = [gen_captions[img_id]]

# Sanity checks
print(f"Number of images in references: {len(gts)}")
print(f"Number of images with generated captions: {len(res)}")

# Compute CIDEr score
cider_scorer = Cider()
score, scores_per_image = cider_scorer.compute_score(gts, res)

# Create a rich console and table
console = Console()
table = Table(title="CIDEr Score per Image")
table.add_column("Image ID", style="cyan", no_wrap=True)
table.add_column("CIDEr Score", style="green")

# Add a row for each image
for img_id, img_score in zip(res.keys(), scores_per_image):
    table.add_row(img_id, f"{img_score:.4f}")

# Add average score row
table.add_row("[bold yellow]Average[/bold yellow]", f"[bold]{score:.4f}[/bold]")

# Print the table
console.print(table)


Number of images in references: 100
Number of images with generated captions: 100


   CIDEr Score per Image    
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━┓
┃ Image ID   ┃ CIDEr Score ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━┩
│ ic-039.jpg │ 0.1202      │
│ ic-005.jpg │ 0.0001      │
│ ic-011.jpg │ 0.0002      │
│ ic-010.jpg │ 0.0290      │
│ ic-004.jpg │ 0.0395      │
│ ic-038.jpg │ 0.0183      │
│ ic-012.jpg │ 0.2865      │
│ ic-006.jpg │ 0.1273      │
│ ic-007.jpg │ 0.0000      │
│ ic-013.jpg │ 0.0192      │
│ ic-017.jpg │ 0.0026      │
│ ic-003.jpg │ 0.0673      │
│ ic-002.jpg │ 0.0000      │
│ ic-016.jpg │ 0.1243      │
│ ic-014.jpg │ 0.0072      │
│ ic-028.jpg │ 0.0001      │
│ ic-029.jpg │ 0.0001      │
│ ic-015.jpg │ 0.0279      │
│ ic-001.jpg │ 0.0053      │
│ ic-066.jpg │ 0.2129      │
│ ic-072.jpg │ 0.0094      │
│ ic-099.jpg │ 0.0062      │
│ ic-098.jpg │ 0.0002      │
│ ic-073.jpg │ 0.0129      │
│ ic-067.jpg │ 0.0750      │
│ ic-059.jpg │ 0.0007      │
│ ic-071.jpg │ 0.0000      │
│ ic-065.jpg │ 0.0032      │
│ ic-064.jpg │ 0.0063      │
│ ic-070.jpg │ 0.0305      │
│ ic-058.jpg │ 0.0109      │
│ ic-100.jpg │ 0.0053      │
│ ic-074.jpg │ 0.0086      │
│ ic-060.jpg │ 0.0060      │
│ ic-048.jpg │ 0.0179      │
│ ic-049.jpg │ 0.0095      │
│ ic-061.jpg │ 1.3014      │
│ ic-075.jpg │ 0.0377      │
│ ic-063.jpg │ 0.0000      │
│ ic-077.jpg │ 0.0001      │
│ ic-088.jpg │ 0.0078      │
│ ic-089.jpg │ 0.0000      │
│ ic-076.jpg │ 0.0404      │
│ ic-062.jpg │ 0.0003      │
│ ic-047.jpg │ 0.0039      │
│ ic-053.jpg │ 0.0039      │
│ ic-084.jpg │ 0.0275      │
│ ic-090.jpg │ 0.0116      │
│ ic-091.jpg │ 0.0064      │
│ ic-085.jpg │ 0.0019      │
│ ic-052.jpg │ 0.0016      │
│ ic-046.jpg │ 0.6656      │
│ ic-078.jpg │ 0.0017      │
│ ic-050.jpg │ 0.0081      │
│ ic-044.jpg │ 0.0000      │
│ ic-093.jpg │ 0.0677      │
│ ic-087.jpg │ 0.0074      │
│ ic-086.jpg │ 0.0005      │
│ ic-092.jpg │ 0.0099      │
│ ic-045.jpg │ 0.1299      │
│ ic-051.jpg │ 0.0437      │
│ ic-079.jpg │ 0.0002      │
│ ic-055.jpg │ 0.0427      │
│ ic-041.jpg │ 0.2397      │
│ ic-069.jpg │ 0.0194      │
│ ic-096.jpg │ 0.0093      │
│ ic-082.jpg │ 0.0085      │
│ ic-083.jpg │ 0.0135      │
│ ic-097.jpg │ 0.0079      │
│ ic-068.jpg │ 0.0073      │
│ ic-040.jpg │ 0.0000      │
│ ic-054.jpg │ 0.0000      │
│ ic-042.jpg │ 0.3013      │
│ ic-056.jpg │ 0.0157      │
│ ic-081.jpg │ 0.1670      │
│ ic-095.jpg │ 0.0001      │
│ ic-094.jpg │ 0.3504      │
│ ic-080.jpg │ 0.0202      │
│ ic-057.jpg │ 0.1487      │
│ ic-043.jpg │ 0.0239      │
│ ic-018.jpg │ 0.0219      │
│ ic-024.jpg │ 0.0008      │
│ ic-030.jpg │ 0.0461      │
│ ic-031.jpg │ 0.0359      │
│ ic-025.jpg │ 0.0439      │
│ ic-019.jpg │ 0.0067      │
│ ic-033.jpg │ 0.0027      │
│ ic-027.jpg │ 0.0076      │
│ ic-026.jpg │ 0.0697      │
│ ic-032.jpg │ 0.1031      │
│ ic-036.jpg │ 0.0644      │
│ ic-022.jpg │ 0.0025      │
│ ic-023.jpg │ 0.0018      │
│ ic-037.jpg │ 0.0005      │
│ ic-021.jpg │ 0.0110      │
│ ic-035.jpg │ 0.0114      │
│ ic-009.jpg │ 0.0421      │
│ ic-008.jpg │ 0.3852      │
│ ic-034.jpg │ 0.0133      │
│ ic-020.jpg │ 0.0101      │
│ Average    │ 0.0593      │
└────────────┴─────────────┘

3.5 Visual Question Answering

In [31]:
import csv
import base64
import requests
import os
import re
import json

# Paths - set these as per your environment
image_folder = "./nlp-hw4-datasets/vqa/images"
csv_file = "./nlp-hw4-datasets/vqa/vqa.csv"
output_csv_file = "vqa_answers.csv"

def parse_streaming_response(response_text):
    lines = response_text.strip().split('\n')
    final_response = None
    for line in lines:
        try:
            data = json.loads(line)
            resp = data.get("response", "").strip()
            if resp:
                final_response = resp  # keep last non-empty response
            if data.get("done", False) and resp:
                break  # stop once final response is found
        except json.JSONDecodeError:
            continue
    return final_response if final_response else None

def encode_image_to_base64(path):
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode()


import re

def parse_options(raw_options_str):
    # Remove outer brackets and whitespace
    s = raw_options_str.strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()

    # Split on pattern: single quote + space + single quote
    parts = re.split(r"'\s+'", s)

    # Clean quotes around each option
    cleaned_options = [p.strip("'").strip('"') for p in parts]

    return cleaned_options


def make_prompt(question, options, labels=["A", "B", "C", "D", "E"]):
    prompt = f"Based on the question below, select the letter (A, B, C, D, or E) corresponding to the correct answer.\n\n"
    prompt += f"Question:\n{question}\n\nOptions:\n"
    for label, option in zip(labels, options):
        prompt += f"{label}. {option}\n"
    prompt += "\nPlease answer only with a single letter: A, B, C, D, or E. If None of the options is correct, answer with 'None'."
    return prompt


def query_ollama(image_b64, prompt, url="http://localhost:11434/api/generate"):
    payload = {
        "model": MODEL,
        "prompt": prompt,
        "image": image_b64,
        "max_tokens": 20,
        "temperature": 0,
        "stream": False,
        "seed": 42,
    }
    response = requests.post(url, json=payload)
    if response.status_code == 200:
        answer = parse_streaming_response(response.text)
        print(f"Raw extracted answer: '{answer}'")  # helpful for debugging
        return answer
    else:
        print(f"Error {response.status_code}: {response.text}")
        return None

results = []

with open(csv_file, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        image_name = row["image"]

        question = row["question"]
        raw_options = row["options"]
        options = parse_options(raw_options)
        
        image_path = os.path.join(image_folder, image_name)
        if not os.path.isfile(image_path):
            print(f"Image not found: {image_path}, skipping...")
            continue

        print(f"Processing {image_name}")

        image_b64 = encode_image_to_base64(image_path)
        prompt = make_prompt(question, options)

        answer = query_ollama(image_b64, prompt)
        print(f"Model Answer: {answer}")

        results.append({
            "image_name": image_name,
            "question": question,
            "model_answer": answer,
            "ground_truth_answer": row["answer"],
            "difficulty": row["difficulty"]
        })



with open(output_csv_file, mode="w", newline="", encoding="utf-8") as f_out:
    writer = csv.DictWriter(f_out, fieldnames=["image_name", "question", "model_answer", "ground_truth_answer", "difficulty"])
    writer.writeheader()
    for row in results:
        writer.writerow(row)

print("All done. Answers saved to", output_csv_file)


Processing vqa-001.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-002.jpg
Raw extracted answer: 'B'
Model Answer: B
Processing vqa-003.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-004.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-005.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-006.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-007.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-008.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-009.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-010.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-011.jpg
Raw extracted answer: 'A'
Model Answer: A
Processing vqa-012.jpg
Raw extracted answer: 'B'
Model Answer: B
Processing vqa-013.jpg
Raw extracted answer: 'B'
Model Answer: B
Processing vqa-014.jpg
Raw extracted answer: 'B'
Model Answer: B
Processing vqa-015.jpg
Raw extracted answer: 'E'
Model Answer: E
Processing vqa-016.jpg
Ra

4.5 Visual Question Answering EVALUATION

In [4]:
import csv

def normalize(text):
    """Normalize text for exact matching."""
    return text.strip().lower()

def evaluate_em_accuracy(results_csv):
    total = 0
    correct = 0
    mismatches = []

    with open(results_csv, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total += 1
            model_ans = normalize(row["model_answer"])
            gt_ans = normalize(row["ground_truth_answer"])
            if model_ans == gt_ans:
                correct += 1
            else:
                mismatches.append({
                    "image": row["image_name"],
                    "question": row["question"],
                    "model_answer": model_ans,
                    "ground_truth": gt_ans
                })

    accuracy = correct / total if total > 0 else 0.0
    return accuracy, mismatches

# Evaluate
em_accuracy, errors = evaluate_em_accuracy("vqa_answers.csv")
print(f"Exact Match Accuracy: {em_accuracy:.2%}")
print(f"Mismatched answers: {len(errors)}")


Exact Match Accuracy: 29.00%
Mismatched answers: 71
